# **Desafio Estatística com Python**
# Frequência e Medidas

### Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## **Configurações Iniciais**

Fonte dos dados: [Kaggle - Netflix Shows and Movies - Exploratory Analysis](https://www.kaggle.com/code/shivamb/netflix-shows-and-movies-exploratory-analysis)

In [ ]:
# Importa as bibliotecas
import pandas as pd
import numpy as np

# Carregamento da base de dados sobre catalogo da Netflix via Google Drive
# Armazena o id do arquivo csv e o insere na variavel da url
sheet_id = '1OJk-yF4ZgqyKP9Q6qgfWziiOQEOCV7mML0p0_AjdWGI'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'

# Le o csv e armazena os dados em um data frame
df = pd.read_csv(url)
df.head()

*Variáveis:*
* `show_id` - id único do filme/série.
* `title` - título do filme ou série
* `director` - diretor do filme ou série
* `cast` - elenco do filme ou série
* `country` - país do filme ou série
* `date_added` - data que foi adicionado no Netflix
* `release_year` - ano de lançamento original do filme
* `rating` - classificação da televisão
* `duration` - duração total do filme ou série.
* `listed_in` - categoria ou gênero do filme ou série.
* `description` - descrição do filme ou série.
* `type` - tipo de filme ou série

## **Exploração Inicial**

*   Quantas linhas e colunas tem o dataset?
*   Quais são os tipos das variáveis e se há valores ausentes?

In [ ]:
# Infos das colunas do df
df.info()

In [ ]:
# Estatisticas gerais do df
df.describe()

In [ ]:
# Informa quantas linhas e colunas
print(f'O dataset tem {df.shape[0]} linhas e {df.shape[1]} colunas.')

# Informa os tipos
def imprimir_checagem_tipos(df):
  texto = ''

  for col_idx in df.dtypes.value_counts().index:
    if texto != '':
      texto = f'{texto} e '

    tipo_categorica = ['object', 'str']

    if col_idx in tipo_categorica:
      texto = f'{texto}categóricas'
    else:
      texto = f'{texto}numéricas'
  return texto

print(f'\nO dataset tem variáveis {imprimir_checagem_tipos(df)}:\n\n{df.dtypes.value_counts()}')

# Informa os nulos
total_nulos = df.isnull().sum()

# Cria a tabela unindo o total absoluto e a porcentagem
tabela_nulos = pd.concat([total_nulos, df.isnull().mean() * 100], axis=1)
tabela_nulos = tabela_nulos.rename(columns={
  0: 'Total de Nulos',
  1: '% de Nulos'
})

# Filtra e mantem apenas as colunas que tem algum nulo, ordenando do maior para o menor
tabela_nulos = tabela_nulos[tabela_nulos['Total de Nulos'] != 0].sort_values('Total de Nulos', ascending=False).round(2)

print(f'\nO dataset contém valores ausentes em {tabela_nulos.shape[0]} colunas.')

# Checa se a tabela nao esta vazia para entao mostra-la
if tabela_nulos.shape[0] > 0:
  tabela_nulos


In [ ]:
# Importa o pacote missingno para visualizacao de dados ausentes
import missingno as msno
msno.matrix(df)

## **Análises de Frequência**

* Qual a proporção de filmes vs. séries no catálogo?
* Qual o gênero mais frequente?

In [ ]:
# Analisa a proporcao de cada tipo de midia (filme ou serie)
total_tipo_midia = df['type'].value_counts() # contagem total de cada tipo de midia
total_tipo_midia_perc = df['type'].value_counts(normalize=True).round(2) * 100 # contagem percentual

# Junta o total e a proporcao de cada tipo de midia num unico df
df_tipo_midia= pd.concat([total_tipo_midia, total_tipo_midia_perc], axis=1)

# Renomeia as colunas
df_tipo_midia.index.name = 'Tipo de mídia'
df_tipo_midia.rename(columns={
    'count': 'Total',
    'proportion': 'Total em %'
  }, inplace=True)

# Imprime resposta
filmes_perc = df_tipo_midia['Total em %'].loc['Movie']
series_perc = df_tipo_midia['Total em %'].loc['TV Show']
print(f'O catálogo possui {filmes_perc}% de filmes e {series_perc}% de séries (razão: {filmes_perc / series_perc:.2f}/1).')

df_tipo_midia

In [ ]:
"""
Funcao que calcula a frequencia absoluta e percentual de uma Series e retorna um DataFrame.
"""
def calcular_frequencia(tabela, idx_txt):
  # Conta a quantidade absoluta de cada elemento
  quantidade = tabela.value_counts().rename('Total')

  # Calcula as porcentagens, arrendondando para 2 casas
  porporcao = tabela.value_counts(normalize=True).rename('Total em %')
  porporcao = (porporcao * 100).round(2)

  # Junta as tabelas das contagens
  df_resultado = pd.concat([quantidade, porporcao], axis=1)
  df_resultado.index.name = idx_txt # Renomeia o index

  return df_resultado

# Armazena o nome da coluna
idx_genero = 'Gênero'

# Extrai cada genero listado em ´listed_in´ para novas linhas
generos_separados = df['listed_in'].str.split(',').explode()

In [ ]:
# Gera o df com as contagens dos generos
df_generos = calcular_frequencia(generos_separados, idx_genero)

# Armazena o genero mais frequente
genero_moda = generos_separados.mode()[0]
# Imprime resposta
print(f'O gênero mais frequente é "{genero_moda}", presente em {df_generos["Total em %"].loc[genero_moda]}% do catálogo.')

df_generos.head()

### Adicional - Limpeza e Tratamento dos Dados de Gênero

- **Remoção de Formatos de Mídia:** Como já tem-se uma análise macro de tipo de mídia, foram retiradas as menções diretas a isso.
- **Fusão de Categorias Redundantes:** Gêneros semanticamente idênticos foram unificados para não fragmentar os volumes reais do catálogo.
- **Eliminação do Viés Geográfico:** Os gêneros com "International" foram retirados porque refletem a perspectiva dos EUA. Fora deste contexto, isso gera distorções (como filmes nacionais sendo classificados como "internacionais"). 
- **Tratamento de Produções Órfãs:** Produções que ficariam sem gênero - rotuladas apenas como "International", "TV Series" ou "Movies" - foram classificadas como "Sem Gênero", garantindo a preservação dos dados e a integridade do volume total da base.

In [ ]:
# Atribui padrao aos elementos que tem apenas o tipo de midia
sem_genero = 'Sem Gênero'

# PRIMEIRA PARTE DA LIMPEZA: SUBSTITUICAO E TRATAMENTO DE ORFAOS

# Dicionario mapeando generos para unificacao
# Generos com 'International' sao mapeados temporariamente para None
mapeamento_genero = {
  'International Movies': None,
  'International TV Shows': None,
  'Movies': sem_genero,
  'TV Shows': sem_genero,
  'Docuseries': 'Documentaries',
  'Stand-Up Comedy & Talk Shows': 'Stand-Up Comedy,Talk Shows',
  'Classic & Cult TV': 'Classic,Cult',
  'Children & Family Movies': 'Children,Family',
  'Kids\' TV' : 'Children',
  'Spanish-Language TV Shows' : 'Spanish',
}

In [ ]:
# Remove espacos e aplica a substituicao do dicionario
df_generos_limpos = generos_separados.str.strip().replace(mapeamento_genero)

# Lista os indexes das producoes sem nenhum genero associado (orfaos)
# 1. .isna() transforma os dados em bool
# 2. .groupby().all() agrupa pelo index e verifica se TODAS as linhas daquela producao sao True (nulas)
lista_ids_vazios = df_generos_limpos.isna().groupby(df_generos_limpos.index).all()
# 3. Filtra apenas onde a condicao deu True e extrai seus indexes
lista_ids_vazios = lista_ids_vazios[lista_ids_vazios].index

# Substitui por 'Sem Genero' apenas os indexes filtrados
df_generos_limpos.loc[df_generos_limpos.index.isin(lista_ids_vazios)] = sem_genero

df_generos_limpos.head()

In [ ]:
# SEGUNDA PARTE DA LIMPEZA: TERMOS REDUNDANTES
# Cria um dicionario dinamico para eliminar palavras redundantes
termos_redundantes = ['Movies', 'TV Shows', 'TV', 'Series', 'Features']
conteudo_limpeza = {}
for palavra in termos_redundantes:
  conteudo_limpeza[palavra] = ''
# Remove as palavras baseadas no dicionario
df_generos_limpos = df_generos_limpos.replace(conteudo_limpeza, regex=True)

# Separa e extrai os generos para novas linhas com tudo higienizado,
# removendo espacos remancescentes apos a separacao
df_generos_limpos = df_generos_limpos.str.split(',').explode().str.strip()

df_generos_limpos.head()

In [ ]:
# Armazena o genero mais frequente - precisa vir antes da funcao
moda_generos_limpos = df_generos_limpos.mode()[0]

# Gera o df com as contagens dos generos tratados
df_generos_limpos = calcular_frequencia(df_generos_limpos, idx_genero)

# Imprime resposta
perc_moda_generos_limpos = df_generos_limpos['Total em %'].loc[moda_generos_limpos]
print(f'Análise Tratada: Após a limpeza, o gênero mais frequente é "{moda_generos_limpos}", representando {perc_moda_generos_limpos}% do catálogo.')

## **Análises Estatísticas**

* Qual a média, mediana e moda do tempo de duração dos
filmes?
* Qual o filme mais curto e mais longo?

## **Visualização de Dados**

* Criar um gráfico de barras para mostrar a quantidade de títulos por gênero.
* Criar um histograma para analisar a distribuição da duração dos
filmes.

## **Atividade Extra**

* Quais são os 5 países que possuem mais produções no catálogo?




In [ ]:
# Separa as producoes para a contagem
df_paises = df['country'].str.split(', ').explode()
# Faz a contagem de cada pais
df_paises_contagem = df_paises.value_counts().rename('Total de produções')

In [ ]:
# Cria o ranking cantinuo com base na contagem, em ordem decrescente
df_paises_ranking = df_paises_contagem.rank(ascending=False, method='dense').rename('Ranking')
# Junta a contagem com o ranking em um só df de classificacao
df_paises_ranking = pd.concat([df_paises_ranking, df_paises_contagem], axis=1)
df_paises_ranking.index.name = 'País'

# Filtra os paises que estao ate a 5 posicao do topo do catalogo
df_paises_ranking[df_paises_ranking['Ranking'] <= 5]